<p align="center">
<a href="https://colab.research.google.com/github/md-ryhan-uddin/ai-lab-experiments/blob/main/experiment_03.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<a href="https://kaggle.com/kernels/welcome?src=https://github.com/md-ryhan-uddin/ai-lab-experiments/blob/main/experiment_03.ipynb">
    <img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle" height="20"/>
</a>
</p>

## **Experiment 03:** Knowledge Representation and Rule-Based Expert System Development using Python

**Course:** ICE 4111: Artificial Intelligence Lab  
**Program:** Bachelor of Science in Information and Communication Engineering (BSc in ICE)  
**Instructor:** Md. Ryhan Uddin

**Student Name:** ______________________________  
**Student ID:** ______________________________  
**Section/Batch:** ______________________________  
**Date of Submission:** ______________________________  


### **Quick Overview**

In this lab you will build a small **rule-based expert system** in Python. You will represent expert knowledge as facts and if-then rules, implement a **forward-chaining inference engine**, and generate human-readable explanations for every conclusion the system reaches. The domain used is academic advising for course load recommendations.

### **Learning Objectives**

After completing this experiment, students will be able to:

- Represent expert knowledge using facts, rules, and explanations.
- Implement a simple forward-chaining inference engine in Python.
- Generate decisions and human-readable justifications from rules.
- Test the expert system on multiple sample cases.
- Explain the difference between forward chaining and backward chaining.
- Identify the strengths and limitations of rule-based reasoning.

## **Theory**

### Introduction

Knowledge representation is the part of AI that stores domain knowledge in a form a computer can reason about. Rule-based expert systems are among the earliest and most interpretable AI systems. They encode expert knowledge as if-then rules and use inference to derive conclusions from facts supplied by the user. Because the reasoning steps are explicit, they are easy to inspect, debug, and explain to non-experts.

### Mathematical Background

A rule-based system can be written as a set of implications of the form

$$p_1 \land p_2 \land \cdots \land p_k \rightarrow q$$

The left-hand side contains conditions and the right-hand side contains a conclusion. In **forward chaining**, the system starts from known facts and repeatedly applies rules whose premises are satisfied. In **backward chaining**, the system starts from a query and checks whether the required facts or subgoals can be proved. This notebook uses forward chaining because it is simple to trace and explain.

### Important Concepts

A typical expert system contains a **knowledge base**, an **inference engine**, and an **explanation module**.

- The knowledge base stores rules and domain facts.
- The inference engine scans the rule set, matches conditions against facts, and adds new conclusions until no more rules fire.
- The explanation module shows which rules were used.

**Conflict resolution** is needed when multiple rules become active at once; common strategies include rule order, priority, or specificity.

```mermaid
flowchart LR
U[User Facts] --> KB[Knowledge Base]
KB --> IE[Inference Engine]
IE --> D[Decision]
IE --> EX[Explanation]
```

### Advantages, Disadvantages, and Applications

**Advantages / Disadvantages**

- Expert systems are highly interpretable and can work well when a domain has clear rules.
- Their disadvantages are limited learning ability, maintenance cost, and difficulty handling noisy or incomplete knowledge.

**Applications**

They are still useful in configuration, diagnosis, tutoring systems, decision support, and compliance checks.

## **Required Software and Libraries**

Install or prepare the following before starting the lab:

- Python 3.12 or later
- Jupyter Notebook, JupyterLab, VS Code, or Google Colab
- Pandas

### Starter Check

This experiment does not require a separate setup-check cell; running the rule-definition cell below and confirming no errors occur means your environment is ready.

## **Dataset Description**

This notebook uses a small **synthetic case set** for academic advising. The knowledge base is hand-written from common prerequisite and workload considerations, which is appropriate for demonstrating rule-based reasoning.

- **Facts:** completed courses, current GPA, programming interest, and workload.
- **Target labels:** recommendation outcomes such as 'safe load', 'moderate load', or 'heavy load warning'.
- **Source:** manually curated rules for teaching purposes.
- **Reason for selection:** the structure is transparent and the inference trace is easy to understand.

## **Experimental Procedure**

Follow the steps below in order:

1. Define the domain vocabulary and the set of base facts.
2. Write rules using if-then logic and attach explanations to each rule.
3. Implement a forward-chaining function that applies all applicable rules.
4. Create sample student profiles and test the system on each profile.
5. Display the inferred conclusion and the explanation trace.
6. Trace one example case in detail to confirm the reasoning is correct.

### Student Checkpoint

Before moving to the discussion section, make sure you can answer:

- Which rules fired for Case A, and in what order?
- Why did Case C trigger a warning instead of a recommendation?
- What would happen if a student satisfied no rule conditions at all?


## **Source Code**

Run the following cells one by one. Read the output after each cell and add comments in your own words where instructed by your teacher.

**Tip:** Do not only run the notebook. Try adding a new rule, changing a threshold (such as the GPA cutoff), or creating a new sample case, and observe what changes.

In [17]:
# Knowledge Base represented using Python Dictionary
from pprint import pprint

knowledge_base = {
    "animal": {
        "has_feathers": "bird",
        "gives_milk": "mammal",
        "lives_in_water": "fish"
    }
}

print("Knowledge Base")
pprint(knowledge_base)

Knowledge Base
{'animal': {'gives_milk': 'mammal',
            'has_feathers': 'bird',
            'lives_in_water': 'fish'}}


In [18]:
# Representative Facts about a bird
facts = {
    "has_feathers": True,
    "can_fly": True,
    "lays_eggs": True
}

print("Known Facts")
for key, value in facts.items():
    print(f"{key} : {value}")

Known Facts
has_feathers : True
can_fly : True
lays_eggs : True


In [19]:
# production rules represented using Python List of Dictionaries
rules = [
    {
        "conditions": ["has_feathers"],
        "conclusion": "bird"
    },
    {
        "conditions": ["gives_milk"],
        "conclusion": "mammal"
    },
    {
        "conditions": ["bird", "can_fly"],
        "conclusion": "flying_bird"
    },
    {
        "conditions": ["bird", "cannot_fly"],
        "conclusion": "flightless_bird"
    }
]

print("Production Rules")
pprint(rules)

Production Rules
[{'conclusion': 'bird', 'conditions': ['has_feathers']},
 {'conclusion': 'mammal', 'conditions': ['gives_milk']},
 {'conclusion': 'flying_bird', 'conditions': ['bird', 'can_fly']},
 {'conclusion': 'flightless_bird', 'conditions': ['bird', 'cannot_fly']}]


In [20]:
# Forward Chaining Algorithm Implementation
def forward_chaining(facts, rules):
    inferred = set()
    changed = True
    while changed:
        changed = False
        for rule in rules:
            if all(facts.get(condition, False) or condition in inferred
                   for condition in rule["conditions"]):
                conclusion = rule["conclusion"]
                if conclusion not in inferred:
                    inferred.add(conclusion)
                    changed = True
                    
    return inferred

In [21]:
# Test the forward chaining algorithm with the given facts and rules
facts = {
    "has_feathers": True,
    "can_fly": True
}

result = forward_chaining(facts, rules)
print("Inferred Knowledge")
print(result)

Inferred Knowledge
{'flying_bird', 'bird'}


In [22]:
facts = {
    "has_feathers": True,
    "cannot_fly": True
}

result = forward_chaining(facts, rules)
print(result)

{'flightless_bird', 'bird'}


### Expert System

In [23]:
# Disease Diagnosis Expert System represented using Python List of Dictionaries
disease_rules = [

    {
        "conditions": ["fever", "cough"],
        "disease": "Flu"
    },
    {
        "conditions": ["fever", "rash"],
        "disease": "Measles"
    },
    {
        "conditions": ["headache", "nausea"],
        "disease": "Migraine"
    }
]

In [24]:
# Function to diagnose disease based on symptoms
def diagnose(symptoms):
    possible = []
    for rule in disease_rules:
        if all(symptom in symptoms for symptom in rule["conditions"]):
            possible.append(rule["disease"])
    return possible

In [25]:
# Example usage of the diagnosis function
patient = [
    "fever",
    "cough"
]

diagnosis = diagnose(patient)

print("Symptoms:", patient)
print("Diagnosis:", diagnosis)

Symptoms: ['fever', 'cough']
Diagnosis: ['Flu']


In [26]:
# User Input for Symptoms
symptoms = []

questions = [
    "fever",
    "cough",
    "rash",
    "headache",
    "nausea"
]

print("Answer using y/n")

for question in questions:
    ans = input(f"Do you have {question}? ")
    if ans.lower() == "y":
        symptoms.append(question)

result = diagnose(symptoms)
print("\nPossible Disease")

if result:
    print(result)
else:
    print("No disease matched.")

Answer using y/n

Possible Disease
['Flu', 'Measles', 'Migraine']


In [27]:
def explain(symptoms):

    print("Reasoning Process")

    for rule in disease_rules:
        if all(symptom in symptoms for symptom in rule["conditions"]):
            print()
            print("Matched Rule")
            print("IF", " AND ".join(rule["conditions"]))
            print("THEN", rule["disease"])

In [28]:
patient = [
    "fever",
    "cough"
]

diagnosis = diagnose(patient)
print("Diagnosis:", diagnosis)
explain(patient)

Diagnosis: ['Flu']
Reasoning Process

Matched Rule
IF fever AND cough
THEN Flu


In [29]:
knowledge = {
    "bird": ["has_feathers"],
    "flying_bird": ["bird", "can_fly"],
    "flightless_bird": ["bird", "cannot_fly"]
}

In [30]:
def backward_chain(goal, facts):

    if goal in facts:
        return True

    if goal not in knowledge:
        return False

    for condition in knowledge[goal]:
        if not backward_chain(condition, facts):
            return False

    return True

In [31]:
facts = {
    "has_feathers",
    "can_fly"
}

goal = "flying_bird"
print("Goal:", goal)
print("Can be proved?")
print(backward_chain(goal, facts))

Goal: flying_bird
Can be proved?
True


## **Expected Output**

After running all cells, the notebook should display:

- The rule base definition without errors.
- The sample student cases as a table.
- A results table showing the fired-rule sequence, the inferred recommendation, and the warning (if any) for each case.
- A detailed step-by-step trace for one example case, including the final facts, the rule trace, and the plain-language explanation.

You should include the important outputs in your lab report and briefly explain what each output means.

## **Viva Questions**

1. **Q:** What is knowledge representation in AI?  
   **A:** It is the process of encoding domain knowledge in a form that a computer can reason about.

2. **Q:** What is an inference engine?  
   **A:** It is the reasoning component that applies rules to facts.

3. **Q:** What is the difference between forward and backward chaining?  
   **A:** Forward chaining starts from facts and derives conclusions, while backward chaining starts from a goal and checks whether it can be proved.

4. **Q:** Why are expert systems interpretable?  
   **A:** Because each conclusion can be traced back to explicit rules.

5. **Q:** What is a rule base?  
   **A:** A collection of if-then statements that encode expert knowledge.

6. **Q:** What is conflict resolution?  
   **A:** The method used to choose which rule to fire when multiple rules are applicable.

7. **Q:** What is a conclusion in a rule system?  
   **A:** A new fact derived after a rule is satisfied.

8. **Q:** Why are rule-based systems useful in education?  
   **A:** They can explain decisions in a clear, human-readable way.

9. **Q:** What is a limitation of rule-based AI?  
   **A:** It does not learn automatically from data.

10. **Q:** Where are expert systems used?  
    **A:** In diagnosis, configuration, compliance checking, and decision support.

## **Exercises**

Complete the following tasks after running the notebook:

1. Add a new rule that recommends a research project track for students with high GPA and ML interest.
2. Modify the inference engine so that it stops after the first matching recommendation rule.
3. Implement backward chaining for one of the conclusions.
4. Attach confidence scores to rules and resolve conflicts using the highest score.
5. Create a second knowledge base for hostel allocation or library access and test it on sample cases.

## **Lab Report Submission Checklist**

Before submitting, confirm that your report includes:

- Completed cover information.
- Short explanation of knowledge representation and forward chaining in your own words.
- The rule base and sample cases used.
- Code and output for the inference engine.
- The results table and the detailed trace for one example case.
- Answers to assigned viva questions.
- Completed exercises.

**Reminder:** Explain what the outputs mean. Do not submit screenshots without interpretation.